In [1]:
import pandas as pd

nav = pd.read_csv("D:/bluestock_mf_capstone/data/raw/02_nav_history.csv", parse_dates=["date"])
nav = nav.sort_values(["amfi_code", "date"]).drop_duplicates(["amfi_code", "date"], keep="first")

def fill_nav(group):
    group = group.set_index("date").sort_index()
    idx = pd.date_range(group.index.min(), group.index.max(), freq="B")
    group = group.reindex(idx)
    group["amfi_code"] = group["amfi_code"].ffill()
    group["nav"] = group["nav"].ffill()
    return group

nav_clean = nav.groupby("amfi_code", group_keys=False).apply(fill_nav).reset_index().rename(columns={"index": "date"})
nav_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_nav.csv", index=False)
invalid_nav = nav_clean[nav_clean["nav"] <= 0]

print(f"Original rows: {len(nav)}")
print(f"Cleaned rows: {len(nav_clean)}")
print(f"Duplicate rows removed: {len(nav) - len(nav_clean)}")
print(f"Invalid NAV rows: {len(invalid_nav)}")

if not invalid_nav.empty:
    print(invalid_nav.head())


Original rows: 46000
Cleaned rows: 46000
Duplicate rows removed: 0
Invalid NAV rows: 0


C:\Users\HARSH GOYAL\AppData\Local\Temp\ipykernel_1848\4135215486.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nav_clean = nav.groupby("amfi_code", group_keys=False).apply(fill_nav).reset_index().rename(columns={"index": "date"})


In [20]:
transactions = pd.read_csv("D:/bluestock_mf_capstone/data/raw/08_investor_transactions.csv", parse_dates=["transaction_date"])

print("Unique transaction types befpre cleaning:")
print(transactions["transaction_type"].unique())
print('Unique KYC statuses before cleaning:', transactions['kyc_status'].unique())


transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date'], infer_datetime_format=True, errors='coerce'
)

# transactions['kyc_status_clean'] = transactions['kyc_status'].astype(str).str.strip().replace({'nan': None})

invalid_amount = transactions[transactions['amount_inr'] <= 0]
invalid_date = transactions[transactions['transaction_date'].isna()]
# invalid_txn_type = transactions[~transactions['transaction_type'].isin({'SIP', 'Lumpsum', 'Redemption'})]
# invalid_kyc = transactions[~transactions['kyc_status_clean'].isin(valid_kyc)]

print(f'Total rows: {len(transactions)}')
print(f'Invalid amount rows: {len(invalid_amount)}')
print(f'Invalid date rows: {len(invalid_date)}')

# print('Unique transaction types after cleaning:', sorted(transactions['transaction_type'].dropna().unique()))
# print('Unique KYC statuses after cleaning:', sorted(transactions['kyc_status_clean'].dropna().unique()))

transactions.to_csv("D:/bluestock_mf_capstone/data/processed/clean_transactions.csv", index=False)

Unique transaction types befpre cleaning:
['SIP' 'Redemption' 'Lumpsum']
Unique KYC statuses before cleaning: ['Verified' 'Pending']
Total rows: 32778
Invalid amount rows: 0
Invalid date rows: 0


C:\Users\HARSH GOYAL\AppData\Local\Temp\ipykernel_1848\1049926236.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  transactions['transaction_date'] = pd.to_datetime(


In [4]:
performance = pd.read_csv("D:/bluestock_mf_capstone/data/raw/07_scheme_performance.csv")

numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct",
]

for col in numeric_cols:
    performance[col] = pd.to_numeric(performance[col], errors="coerce")

invalid_numeric = performance[performance[numeric_cols].isna().any(axis=1)]
negative_sharpe = performance[performance["sharpe_ratio"] < 0]
invalid_expense = performance[~performance["expense_ratio_pct"].between(0.1, 2.5, inclusive="both")]

print(f"Total performance rows: {len(performance)}")
print(f"Rows with non-numeric performance values: {len(invalid_numeric)}")
print(f"Rows with negative Sharpe ratio: {len(negative_sharpe)}")
print(f"Rows with expense ratio outside 0.1%-2.5%: {len(invalid_expense)}")

if not invalid_numeric.empty:
    print("\nExample invalid numeric rows:")
    print(invalid_numeric.head())

if not negative_sharpe.empty:
    print("\nExample negative Sharpe rows:")
    print(negative_sharpe[["amfi_code", "scheme_name", "sharpe_ratio"]].head())

if not invalid_expense.empty:
    print("\nExample invalid expense ratio rows:")
    print(invalid_expense[["amfi_code", "scheme_name", "expense_ratio_pct"]].head())

performance_clean = performance.copy()
performance_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_performance.csv", index=False)


Total performance rows: 40
Rows with non-numeric performance values: 0
Rows with negative Sharpe ratio: 0
Rows with expense ratio outside 0.1%-2.5%: 0


In [5]:
aum = pd.read_csv("D:/bluestock_mf_capstone/data/raw/03_aum_by_fund_house.csv")

aum["date"] = pd.to_datetime(aum["date"], errors="coerce", infer_datetime_format=True)
aum["aum_lakh_crore"] = pd.to_numeric(aum["aum_lakh_crore"], errors="coerce")
aum["aum_crore"] = pd.to_numeric(aum["aum_crore"], errors="coerce")
aum["num_schemes"] = pd.to_numeric(aum["num_schemes"], errors="coerce", downcast="integer")

aum = aum.drop_duplicates(subset=["date", "fund_house"], keep="first")

aum_nulls = aum[aum["date"].isna() | aum["fund_house"].isna() | aum[["aum_lakh_crore", "aum_crore", "num_schemes"]].isna().any(axis=1)]

aum_clean = aum.dropna(subset=["date", "fund_house", "aum_lakh_crore", "aum_crore", "num_schemes"]).reset_index(drop=True)

print(f"Total AUM rows: {len(aum)}")
print(f"Duplicate AUM rows removed: {len(aum) - len(aum.drop_duplicates(subset=['date', 'fund_house']))}")
print(f"Rows with null or invalid values: {len(aum_nulls)}")
print(f"Clean AUM rows: {len(aum_clean)}")

if not aum_nulls.empty:
    print("\nExample invalid AUM rows:")
    print(aum_nulls.head())

aum_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_aum.csv", index=False)


Total AUM rows: 90
Duplicate AUM rows removed: 0
Rows with null or invalid values: 0
Clean AUM rows: 90


C:\Users\HARSH GOYAL\AppData\Local\Temp\ipykernel_1848\3019482121.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  aum["date"] = pd.to_datetime(aum["date"], errors="coerce", infer_datetime_format=True)


In [ ]:
sip = pd.read_csv("D:/bluestock_mf_capstone/data/raw/04_monthly_sip_inflows.csv")

sip["month"] = pd.to_datetime(sip["month"], errors="coerce", infer_datetime_format=True)

sip = sip.drop_duplicates()

sip_nulls = sip[sip[["month"] + numeric_cols].isna().any(axis=1)]

sip_clean = sip.dropna(subset=["month"] + numeric_cols).reset_index(drop=True)

print(f"Total SIP rows: {len(sip)}")
print(f"Duplicate SIP rows removed: {len(sip) - len(sip.drop_duplicates(keep='first'))}")
print(f"Rows with null or invalid values: {len(sip_nulls)}")
print(f"Clean SIP rows: {len(sip_clean)}")

if not sip_nulls.empty:
    print("\nExample invalid SIP rows:")
    print(sip_nulls.head())

sip_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_sip.csv", index=False)


Total SIP rows: 48
Duplicate SIP rows removed: 0
Rows with null or invalid values: 12
Clean SIP rows: 36

Example invalid SIP rows:
       month  sip_inflow_crore  active_sip_accounts_crore  \
0 2022-01-01             11517                       4.91   
1 2022-02-01             11438                       4.93   
2 2022-03-01             12328                       5.09   
3 2022-04-01             11863                       5.48   
4 2022-05-01             12286                       5.55   

   new_sip_accounts_lakh  sip_aum_lakh_crore  yoy_growth_pct  
0                   9.10                4.80             NaN  
1                   8.20                4.85             NaN  
2                  10.50                5.01             NaN  
3                   9.52                5.12             NaN  
4                   8.10                5.15             NaN  


C:\Users\HARSH GOYAL\AppData\Local\Temp\ipykernel_1848\3927429876.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  sip["month"] = pd.to_datetime(sip["month"], errors="coerce", infer_datetime_format=True)


In [9]:
inflows = pd.read_csv("D:/bluestock_mf_capstone/data/raw/05_category_inflows.csv")

# inflows["month"] = pd.to_datetime(inflows["month"], errors="coerce", infer_datetime_format=True)

inflows = inflows.drop_duplicates(keep="first")

inflows_nulls = inflows[inflows[["month", "category", "net_inflow_crore"]].isna().any(axis=1)]

inflows_clean = inflows.dropna(subset=["month", "category", "net_inflow_crore"]).reset_index(drop=True)

print(f"Total inflows rows: {len(inflows)}")
print(f"Duplicate inflows rows removed: {len(inflows) - len(inflows.drop_duplicates(keep='first'))}")
print(f"Rows with null or invalid values: {len(inflows_nulls)}")
print(f"Clean inflows rows: {len(inflows_clean)}")

if not inflows_nulls.empty:
    print("\nExample invalid inflows rows:")
    print(inflows_nulls.head())

inflows_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_category_inflows.csv", index=False)


Total inflows rows: 144
Duplicate inflows rows removed: 0
Rows with null or invalid values: 0
Clean inflows rows: 144


In [10]:
folio = pd.read_csv("D:/bluestock_mf_capstone/data/raw/06_industry_folio_count.csv")

folio_numeric_cols = [
    "total_folios_crore",
    "equity_folios_crore",
    "debt_folios_crore",
    "hybrid_folios_crore",
    "others_folios_crore",
]

folio = folio.drop_duplicates(keep="first")

folio_nulls = folio[folio[["month"] + folio_numeric_cols].isna().any(axis=1)]

folio_clean = folio.dropna(subset=["month"] + folio_numeric_cols).reset_index(drop=True)

print(f"Total folio rows: {len(folio)}")
print(f"Duplicate folio rows removed: {len(folio) - len(folio.drop_duplicates(keep='first'))}")
print(f"Rows with null or invalid values: {len(folio_nulls)}")
print(f"Clean folio rows: {len(folio_clean)}")

if not folio_nulls.empty:
    print("\nExample invalid folio rows:")
    print(folio_nulls.head())

folio_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_folio_count.csv", index=False)


Total folio rows: 21
Duplicate folio rows removed: 0
Rows with null or invalid values: 0
Clean folio rows: 21


In [11]:
holdings = pd.read_csv("D:/bluestock_mf_capstone/data/raw/09_portfolio_holdings.csv")

holdings = holdings.drop_duplicates(keep="first")

holdings_nulls = holdings[holdings[["amfi_code", "stock_symbol", "stock_name", "sector", "portfolio_date", "weight_pct", "market_value_cr", "current_price_inr"]].isna().any(axis=1)]

holdings_clean = holdings.dropna(subset=["amfi_code", "stock_symbol", "stock_name", "sector", "portfolio_date", "weight_pct", "market_value_cr", "current_price_inr"]).reset_index(drop=True)

print(f"Total holdings rows: {len(holdings)}")
print(f"Duplicate holdings rows removed: {len(holdings) - len(holdings.drop_duplicates(keep='first'))}")
print(f"Rows with null or invalid values: {len(holdings_nulls)}")
print(f"Clean holdings rows: {len(holdings_clean)}")

if not holdings_nulls.empty:
    print("\nExample invalid holdings rows:")
    print(holdings_nulls.head())

holdings_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_holdings.csv", index=False)


Total holdings rows: 322
Duplicate holdings rows removed: 0
Rows with null or invalid values: 0
Clean holdings rows: 322


In [12]:
indices = pd.read_csv("D:/bluestock_mf_capstone/data/raw/10_benchmark_indices.csv")

indices = indices.drop_duplicates(keep="first")

indices_nulls = indices[indices[["date", "index_name", "close_value"]].isna().any(axis=1)]

indices_clean = indices.dropna(subset=["date", "index_name", "close_value"]).reset_index(drop=True)

print(f"Total indices rows: {len(indices)}")
print(f"Duplicate indices rows removed: {len(indices) - len(indices.drop_duplicates(keep='first'))}")
print(f"Rows with null or invalid values: {len(indices_nulls)}")
print(f"Clean indices rows: {len(indices_clean)}")

if not indices_nulls.empty:
    print("\nExample invalid indices rows:")
    print(indices_nulls.head())

indices_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_indices.csv", index=False)


Total indices rows: 8050
Duplicate indices rows removed: 0
Rows with null or invalid values: 0
Clean indices rows: 8050


In [13]:
fm = pd.read_csv("D:/bluestock_mf_capstone/data/raw/01_fund_master.csv")

numeric_cols = [
    "expense_ratio_pct",
    "exit_load_pct",
    "min_sip_amount",
    "min_lumpsum_amount",
]

fm = fm.drop_duplicates(keep="first")

fm_nulls = fm[fm[["amfi_code", "fund_house", "scheme_name", "category", "sub_category", "plan", "launch_date", "benchmark", "expense_ratio_pct", "exit_load_pct", "min_sip_amount", "min_lumpsum_amount", "fund_manager", "risk_category", "sebi_category_code"]].isna().any(axis=1)]

fm_clean = fm.dropna(subset=["amfi_code", "fund_house", "scheme_name", "category", "sub_category", "plan", "launch_date", "benchmark", "expense_ratio_pct", "exit_load_pct", "min_sip_amount", "min_lumpsum_amount", "fund_manager", "risk_category", "sebi_category_code"]).reset_index(drop=True)

print(f"Total FM rows: {len(fm)}")
print(f"Duplicate FM rows removed: {len(fm) - len(fm.drop_duplicates(keep='first'))}")
print(f"Rows with null or invalid values: {len(fm_nulls)}")
print(f"Clean FM rows: {len(fm_clean)}")

if not fm_nulls.empty:
    print("\nExample invalid FM rows:")
    print(fm_nulls.head())

fm_clean.to_csv("D:/bluestock_mf_capstone/data/processed/clean_fund_master.csv", index=False)


Total FM rows: 40
Duplicate FM rows removed: 0
Rows with null or invalid values: 0
Clean FM rows: 40
